# LTX Video API Google Colab GPU Backend

Notebook này cho phép khởi chạy Backend API tạo video từ văn bản (sử dụng mô hình **LTX-Video FP8**) trên GPU miễn phí của Google Colab.
Rất hữu ích để tận dụng GPU T4 miễn phí thay vì chạy trên máy cá nhân không có VGA mạnh.

### Hướng dẫn sử dụng:
1. Cập nhật mã nguồn: Hãy đảm bảo bạn đã đẩy thư mục `ltxcolab` lên tài khoản GitHub của mình (ví dụ: `https://github.com/YOUR_USERNAME/ltxcolab.git`) và cập nhật link ở ô code thứ 1.
2. Truy cập menu **Runtime** -> **Change runtime type** -> Chọn GPU (T4) làm Hardware accelerator.
3. Nhấn nút **Chạy tất cả (Run all)** hoặc chạy từng ô code bên dưới theo thứ tự.
4. Chờ cho đến khi Cloudflare Tunnel khởi tạo xong, hệ thống sẽ in ra một liên kết dạng `https://xxx.trycloudflare.com`.
5. Sử dụng địa chỉ này cho frontend hoặc ứng dụng gọi API của bạn.

### Các API chính:
- Tải model: `POST {URL}/model/download`
- Tạo video: `POST {URL}/generate` (truyền form-data: `text`, `num_inference_steps`...)


In [ ]:
#@title 1. Cài đặt các gói thư viện phụ thuộc (Dependencies)
import os
import sys

print("--- 1. Đang tải mã nguồn ứng dụng từ GitHub... ---")
# TODO: Thay đổi đường dẫn GitHub bên dưới thành repo chứa mã nguồn của bạn
GIT_REPO_URL = "https://github.com/akavipno01/ltxcolab.git"

if not os.path.exists("/content/ltxcolab"):
    !git clone $GIT_REPO_URL /content/ltxcolab

%cd /content/ltxcolab

print("\n--- 2. Đang cài đặt các thư viện cần thiết... ---")
!pip install -r backend/requirements.txt

print("\n--- 3. Tải xuống Cloudflare Tunnel để tạo đường truyền kết nối công khai... ---")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("\nHoàn tất bước chuẩn bị môi trường!")

In [ ]:
#@title 2. Tải trước mô hình AI LTX-Video FP8
from huggingface_hub import hf_hub_download
import json
from pathlib import Path

model_dir = Path("/content/data/models/ltx-video")
model_id = "Lightricks/LTX-Video"
filename = "ltxv-2b-0.9.8-distilled-fp8.safetensors"
marker_path = model_dir / ".ltx-video-model.json"

if not marker_path.is_file():
    print(f"Đang tải mô hình {filename} từ HuggingFace về bộ nhớ đệm Colab...")
    model_dir.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id=model_id,
        filename=filename,
        local_dir=str(model_dir)
    )
    marker_path.write_text(json.dumps({"model_id": model_id}, ensure_ascii=False), encoding="utf-8")
    print("\nTải mô hình thành công!")
else:
    print("Mô hình đã được tải và sẵn sàng sử dụng.")

In [ ]:
#@title 3. Khởi chạy Backend và tạo đường truyền liên kết kết nối
import subprocess
import time
import re
import sys

# Đảm bảo tạo sẵn thư mục lưu trữ data
data_dir = "/content/data"
os.makedirs(os.path.join(data_dir, "outputs"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "temp"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "models"), exist_ok=True)

# Khởi động FastAPI Backend (uvicorn)
print("Đang khởi động backend LTX-Video...")
backend_process = subprocess.Popen(
    ["python", "run.py"],
    cwd="/content/ltxcolab/backend",
    env={
        **os.environ,
        "LTX_VIDEO_DATA_DIR": data_dir
    }
)

# Đợi backend uvicorn chạy (khoảng 3 giây)
time.sleep(3)

# Chạy Cloudflare Tunnel
print("Đang khởi tạo Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

colab_url = None
try:
    while True:
        line = tunnel_process.stdout.readline()
        if not line:
            break
        if "trycloudflare.com" in line or "error" in line.lower() or "tunnel" in line.lower():
            print("[Cloudflared]", line.strip())
        
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            colab_url = match.group(0)
            print("\n" + "="*60)
            print(f"🎉 KHỞI CHẠY GOOGLE COLAB BACKEND THÀNH CÔNG!")
            print(f"🔗 Địa chỉ API Colab của bạn là:")
            print(f"   {colab_url}")
            print("\n============================================================")
            print("📌 API tạo video (POST): {colab_url}/generate")
            print("="*60 + "\n")
            
    # Giữ cho tiến trình chạy để tiếp tục phục vụ
    backend_process.wait()
except KeyboardInterrupt:
    print("\nĐang dừng các tiến trình...")
    tunnel_process.terminate()
    backend_process.terminate()
    print("Đã dừng.")
